# Estudo de Estacionaridade das Séries do PIB-Nowcast

Objetivo: testar diferentes especificações de transformação para cada variável do modelo e identificar a mais parcimoniosa que atinge estacionaridade.

> **Disclaimer:** As transformações abaixo são aplicadas sobre as séries **já com ajuste sazonal**. Variáveis que possuem ajuste sazonal de origem (da fonte) são usadas diretamente; as demais passam pelo ajuste sazonal via X-13ARIMA-SEATS conforme implementado na função `seas_adj`.

**Transformações candidatas:**
- Log, Box-Cox, Yeo-Johnson
- 1ª diferença, diferença sazonal (12 para mensal, 4 para trimestral)
- Combinações cumulativas

**Critério:** maioria dos testes (ADF, KPSS, PP, DFGLS) indica estacionaridade. Em caso de empate entre especificações, a mais parcimoniosa vence.

> **Nota:** A variável `pib` usa variação percentual YoY (4 trimestres), **não** dlog4.

In [1]:
# %% Setup
import warnings
warnings.filterwarnings('ignore')

import sys
from pathlib import Path
for pr in [Path.cwd(), Path.cwd().parent]:
    if (pr / 'pib_nowcast').is_dir():
        sys.path.insert(0, str(pr))
        break

import numpy as np
import pandas as pd

from pib_nowcast.config import SERIES_SPEC, LAST_DATA, DATA_DIR
from pib_nowcast.utils.transformations import (
    stationarity_tests,
    is_stationary,
    seas_adj,
    PIPELINE_REGISTRY,
    PIPELINE_NAME_TO_ID,
    MONTHLY_PIPELINE_IDS,
    QUARTERLY_PIPELINE_IDS,
    apply_transform_pipeline,
)

In [2]:
# %% Carregar dados (já com ajuste sazonal)
specs_df = pd.read_csv(SERIES_SPEC, sep=';')
full_data = pd.read_excel(LAST_DATA, sheet_name='full_dataset', index_col='Date')
full_data = seas_adj(full_data, specs_df)

# Frequência por variável
freq_map = specs_df.set_index('variable')['frequency'].to_dict()

print(f"Variáveis: {list(full_data.columns)}")
print(f"Período: {full_data.index.min()} a {full_data.index.max()}")
print(f"Shape: {full_data.shape}")
full_data.head()

Variáveis: ['ind_extrativa', 'ind_transformacao', 'ind_bens_capital', 'ind_bens_intermediarios', 'ind_bens_consumo', 'ind_construcao', 'cons_energ_comercial', 'cons_energ_industrial', 'pmc_ampliada', 'ibcbr_agro', 'ibcbr_ind', 'ibcbr_servicos', 'icc_fecomercio', 'icea_fecomercio', 'ief_fecomercio', 'ics_fgv', 'isas_fgv', 'ies_fgv', 'caged_estoque', 'renda_media', 'tx_desemprego', 'ipca_12m', 'ipca15', 'igpm', 'igpdi', 'incc', 'ipam', 'pib']
Período: 1996-01-01 00:00:00 a 2026-05-01 00:00:00
Shape: (365, 28)


,ind_extrativa,ind_transformacao,ind_bens_capital,ind_bens_intermediarios,ind_bens_consumo,ind_construcao,cons_energ_comercial,cons_energ_industrial,pmc_ampliada,ibcbr_agro,...,caged_estoque,renda_media,tx_desemprego,ipca_12m,ipca15,igpm,igpdi,incc,ipam,pib
Date,,,,,,,,,,,,,,,,,,,,,
1996-01-01,NaN,NaN,NaN,NaN,NaN,NaN,2740.351755,9470.107161,NaN,NaN,...,20510321.16,NaN,NaN,22.367568,NaN,0.822309,0.949714,1.700387,0.548279,NaN
1996-02-01,NaN,NaN,NaN,NaN,NaN,NaN,2817.905495,9538.658463,NaN,NaN,...,20478142.98,NaN,NaN,20.823798,NaN,1.129591,1.072415,0.154249,0.890830,NaN
1996-03-01,NaN,NaN,NaN,NaN,NaN,NaN,2871.389566,9526.253487,NaN,NaN,...,20410925.09,NaN,NaN,19.128104,NaN,0.427530,0.312773,0.723082,0.500904,96.84
1996-04-01,NaN,NaN,NaN,NaN,NaN,NaN,2886.970413,9433.149315,NaN,NaN,...,20386651.46,NaN,NaN,17.662601,NaN,0.524197,0.974255,0.753334,0.224605,NaN
1996-05-01,NaN,NaN,NaN,NaN,NaN,NaN,2890.245308,9505.022225,NaN,NaN,...,20384644.76,NaN,NaN,16.881171,NaN,1.667012,1.868364,1.225200,1.844305,NaN


## Registro de Pipelines de Transformação

Cada pipeline possui um **ID numérico** fixo definido em `PIPELINE_REGISTRY`. A tabela abaixo lista todos os pipelines disponíveis.

In [3]:
# %% Tabela de referência dos pipelines
reg_rows = []
for pid, (name, steps, needs_pos) in sorted(PIPELINE_REGISTRY.items()):
    grupo = 'Trimestral' if pid >= 13 else 'Mensal'
    if pid in MONTHLY_PIPELINE_IDS and pid in QUARTERLY_PIPELINE_IDS:
        grupo = 'Ambos'
    reg_rows.append({
        'pipeline_id': pid,
        'pipeline': name,
        'n_steps': len(steps),
        'requer_positivo': '✓' if needs_pos else '',
        'frequência': grupo,
    })

pd.DataFrame(reg_rows)

,pipeline_id,pipeline,n_steps,requer_positivo,frequência
0,0,nível,0,,Ambos
1,1,diff,1,,Ambos
2,2,log,1,✓,Ambos
3,3,log→diff,2,✓,Ambos
4,4,boxcox,1,✓,Mensal
5,5,boxcox→diff,2,✓,Mensal
6,6,yeojohnson,1,,Mensal
7,7,yeojohnson→diff,2,,Mensal
8,8,sdiff12,1,,Mensal
9,9,sdiff12→diff,2,,Mensal


## Teste de todas as combinações variável × pipeline

In [4]:
# %% Loop por variável
all_results = []

for col in full_data.columns:
    raw = full_data[col].dropna()
    freq = freq_map.get(col, 'Monthly')
    pipe_ids = QUARTERLY_PIPELINE_IDS if freq == 'Quaterly' else MONTHLY_PIPELINE_IDS

    for pid in pipe_ids:
        pipe_name, steps, _ = PIPELINE_REGISTRY[pid]
        transformed = apply_transform_pipeline(raw, pid)

        if transformed is None:
            all_results.append({
                'variable': col,
                'frequency': freq,
                'pipeline_id': pid,
                'pipeline': pipe_name,
                'n_steps': len(steps),
                'n_obs': 0,
                'ADF': None,
                'KPSS': None,
                'Phillips-Perron': None,
                'DFGLS': None,
                'n_stationary': 0,
                'is_stationary': False,
                'applicable': False,
            })
            continue

        try:
            tests_df = stationarity_tests(transformed)
            test_results = tests_df.set_index('test')['is_stationary'].to_dict()
            # Contar apenas testes que rodaram (não None)
            valid_results = {k: v for k, v in test_results.items() if v is not None}
            n_stat = sum(valid_results.values())
            n_valid = len(valid_results)
            is_stat = n_stat > n_valid / 2 if n_valid > 0 else False
        except Exception as e:
            print(f'  ⚠ {col}/{pipe_name}: erro nos testes ({type(e).__name__}), pulando.')
            all_results.append({
                'variable': col,
                'frequency': freq,
                'pipeline_id': pid,
                'pipeline': pipe_name,
                'n_steps': len(steps),
                'n_obs': len(transformed),
                'ADF': None,
                'KPSS': None,
                'Phillips-Perron': None,
                'DFGLS': None,
                'n_stationary': 0,
                'is_stationary': False,
                'applicable': False,
            })
            continue

        all_results.append({
            'variable': col,
            'frequency': freq,
            'pipeline_id': pid,
            'pipeline': pipe_name,
            'n_steps': len(steps),
            'n_obs': len(transformed),
            'ADF': test_results.get('ADF'),
            'KPSS': test_results.get('KPSS'),
            'Phillips-Perron': test_results.get('Phillips-Perron'),
            'DFGLS': test_results.get('DFGLS'),
            'n_stationary': n_stat,
            'is_stationary': is_stat,
            'applicable': True,
        })

    print(f'✓ {col}')

results_df = pd.DataFrame(all_results)
print(f'\nTotal de combinações testadas: {len(results_df)}')
print(f'Inaplicáveis: {(~results_df["applicable"]).sum()}')

✓ ind_extrativa


✓ ind_transformacao


✓ ind_bens_capital


✓ ind_bens_intermediarios


✓ ind_bens_consumo


...

✓ tx_desemprego


✓ ipca_12m
✓ ipca15


✓ igpm
✓ igpdi


✓ incc
✓ ipam


✓ pib

Total de combinações testadas: 360
Inaplicáveis: 35


## Tabela Resumo

Para cada variável e pipeline, mostra se cada teste indicou estacionaridade (✓/✗) e o consenso geral.

In [5]:
# %% Tabela resumo formatada

def fmt_bool(v):
    if v is None:
        return '—'
    return '✓' if v else '✗'

display_df = results_df.copy()
for test_col in ['ADF', 'KPSS', 'Phillips-Perron', 'DFGLS', 'is_stationary']:
    display_df[test_col] = display_df[test_col].map(fmt_bool)

display_df = display_df.rename(columns={
    'variable': 'Variável',
    'pipeline_id': 'ID',
    'pipeline': 'Pipeline',
    'n_steps': 'Etapas',
    'n_obs': 'N obs',
    'n_stationary': 'Votos',
    'is_stationary': 'Estacionária?',
    'applicable': 'Aplicável',
})

display_df[['Variável', 'ID', 'Pipeline', 'Etapas', 'N obs', 'ADF', 'KPSS', 'Phillips-Perron', 'DFGLS', 'Votos', 'Estacionária?']]

,Variável,ID,Pipeline,Etapas,N obs,ADF,KPSS,Phillips-Perron,DFGLS,Votos,Estacionária?
0,ind_extrativa,0,nível,0,292,✗,✗,—,—,0,✗
1,ind_extrativa,1,diff,1,291,✓,✓,—,—,2,✓
2,ind_extrativa,2,log,1,292,✓,✗,—,—,1,✗
3,ind_extrativa,3,log→diff,2,291,✓,✓,—,—,2,✓
4,ind_extrativa,4,boxcox,1,292,✗,✗,—,—,0,✗
...,...,...,...,...,...,...,...,...,...,...,...
355,pib,13,yoy_pct,1,117,✗,✓,—,—,1,✗
356,pib,14,sdiff4,1,117,✗,✓,—,—,1,✗
357,pib,15,sdiff4→diff,2,116,✓,✓,—,—,2,✓
358,pib,16,log→sdiff4,2,117,✗,✓,—,—,1,✗


## Recomendação

Para cada variável, selecionar a transformação **mais parcimoniosa** (menor nº de etapas) dentre as estacionárias.

Em caso de empate de parcimônia, preferências:
1. `log` sobre `boxcox`/`yeojohnson` (mais interpretável)
2. Mais votos de estacionaridade
3. Mais observações preservadas

In [6]:
# %% Selecionar melhor pipeline por variável

# Ordem de preferência para desempate (menor = melhor)
PIPELINE_PREFERENCE = {
    # Mensais
    'nível': 0,
    'diff': 1,
    'log': 2,
    'log→diff': 3,
    'sdiff12': 4,
    'log→sdiff12': 5,
    'yeojohnson': 6,
    'yeojohnson→diff': 7,
    'boxcox': 8,
    'boxcox→diff': 9,
    'sdiff12→diff': 10,
    'log→sdiff12→diff': 11,
    'boxcox→sdiff12→diff': 12,
    # Trimestrais
    'yoy_pct': 1,
    'sdiff4': 4,
    'sdiff4→diff': 10,
    'log→sdiff4': 5,
    'log→sdiff4→diff': 11,
}

stationary = results_df[results_df['is_stationary'] & results_df['applicable']].copy()
stationary['preference'] = stationary['pipeline'].map(PIPELINE_PREFERENCE).fillna(99)

# Ordenar: menos etapas → mais preferido → mais votos → mais obs
stationary = stationary.sort_values(
    ['n_steps', 'preference', 'n_stationary', 'n_obs'],
    ascending=[True, True, False, False],
)

# Melhor por variável
best = stationary.groupby('variable').first().reset_index()
best = best[['variable', 'frequency', 'pipeline_id', 'pipeline', 'n_steps', 'n_obs', 'n_stationary']]
best.columns = ['Variável', 'Frequência', 'Pipeline ID', 'Pipeline Recomendado', 'Etapas', 'N obs', 'Votos']

print('═' * 80)
print('  RECOMENDAÇÃO DE TRANSFORMAÇÃO POR VARIÁVEL')
print('═' * 80)
best

════════════════════════════════════════════════════════════════════════════════
  RECOMENDAÇÃO DE TRANSFORMAÇÃO POR VARIÁVEL
════════════════════════════════════════════════════════════════════════════════


,Variável,Frequência,Pipeline ID,Pipeline Recomendado,Etapas,N obs,Votos
0,caged_estoque,Monthly,1,diff,1,363,2
1,cons_energ_comercial,Monthly,1,diff,1,363,2
2,cons_energ_industrial,Monthly,1,diff,1,363,2
3,ibcbr_agro,Monthly,1,diff,1,279,2
4,ibcbr_ind,Monthly,1,diff,1,279,2
5,ibcbr_servicos,Monthly,1,diff,1,279,2
6,icc_fecomercio,Monthly,1,diff,1,324,2
7,icea_fecomercio,Monthly,1,diff,1,324,2
8,ics_fgv,Monthly,1,diff,1,214,2
9,ief_fecomercio,Monthly,0,nível,0,325,2


In [7]:
# %% Variáveis sem nenhuma transformação estacionária
all_vars = set(full_data.columns)
stationary_vars = set(stationary['variable'].unique())
problematic = all_vars - stationary_vars

if problematic:
    print('⚠️  Variáveis sem NENHUMA transformação estacionária:')
    for v in sorted(problematic):
        print(f'   - {v}')
else:
    print('✅ Todas as variáveis possuem pelo menos uma transformação estacionária.')

✅ Todas as variáveis possuem pelo menos uma transformação estacionária.


In [8]:
# %% Resumo por variável: todas as opções estacionárias
for var in full_data.columns:
    var_results = stationary[stationary['variable'] == var]
    if var_results.empty:
        print(f"\n{'='*60}")
        print(f'  {var}: NENHUMA transformação estacionária')
        print(f"{'='*60}")
        continue

    print(f"\n{'='*60}")
    print(f'  {var} — {len(var_results)} opções estacionárias')
    print(f"  Recomendado: [{var_results.iloc[0]['pipeline_id']}] {var_results.iloc[0]['pipeline']}")
    print(f"{'='*60}")
    for _, row in var_results.iterrows():
        marker = ' ★' if row.name == var_results.index[0] else '  '
        print(f"{marker} [{int(row['pipeline_id']):>2d}] {row['pipeline']:<25s}  votos={int(row['n_stationary'])}/4  obs={int(row['n_obs'])}")


  ind_extrativa — 9 opções estacionárias
  Recomendado: [1] diff
 ★ [ 1] diff                       votos=2/4  obs=291
   [ 8] sdiff12                    votos=2/4  obs=280
   [ 3] log→diff                   votos=2/4  obs=291
   [10] log→sdiff12                votos=2/4  obs=280
   [ 7] yeojohnson→diff            votos=2/4  obs=291
   [ 5] boxcox→diff                votos=2/4  obs=291
   [ 9] sdiff12→diff               votos=2/4  obs=279
   [11] log→sdiff12→diff           votos=2/4  obs=279
   [12] boxcox→sdiff12→diff        votos=2/4  obs=279

  ind_transformacao — 9 opções estacionárias
  Recomendado: [1] diff
 ★ [ 1] diff                       votos=2/4  obs=291
   [ 8] sdiff12                    votos=2/4  obs=280
   [ 3] log→diff                   votos=2/4  obs=291
   [10] log→sdiff12                votos=2/4  obs=280
   [ 7] yeojohnson→diff            votos=2/4  obs=291
   [ 5] boxcox→diff                votos=2/4  obs=291
   [ 9] sdiff12→diff               votos=2/4  obs=279


## Exportação CSV

Salva os resultados completos e a recomendação por variável em arquivos CSV.

In [9]:
# %% Exportar resultados
csv_dir = DATA_DIR / 'stationarity'
csv_dir.mkdir(parents=True, exist_ok=True)

# Resultados completos
results_df.to_csv(csv_dir / 'stationarity_results.csv', index=False, sep=';')
print(f'✓ Resultados completos salvos em {csv_dir / "stationarity_results.csv"}')

# Recomendação por variável
best_export = stationary.groupby('variable').first().reset_index()
best_export = best_export[['variable', 'frequency', 'pipeline_id', 'pipeline', 'n_steps', 'n_obs', 'n_stationary']]
best_export.to_csv(csv_dir / 'stationarity_best.csv', index=False, sep=';')
print(f'✓ Recomendações salvas em {csv_dir / "stationarity_best.csv"}')

✓ Resultados completos salvos em C:\Users\lisbo\OneDrive\Documentos\PIB-Nowcast\pib_nowcast\data\stationarity\stationarity_results.csv
✓ Recomendações salvas em C:\Users\lisbo\OneDrive\Documentos\PIB-Nowcast\pib_nowcast\data\stationarity\stationarity_best.csv
